# Import Libraries

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Load data

In [0]:
df = spark.table("dataset.default.`1785258801617_sales_case_study_2021_1_1`").toPandas()
df['Date'] = pd.to_datetime(df['Date'])

In [0]:
df = spark.table("dataset.default.`1785258801617_sales_case_study_2021_1_1`").toPandas()
df.head() # Shows first 5 rows

,Date,Sales,Cost Of Sales,Quantity Sold
0,2013-12-30,223937.967869726530,230079.621012448040,6827
1,2013-12-31,300345.484563985660,306986.120484283190,9268
2,2014-01-01,86782.467727417796,87986.318214449886,2678
3,2014-01-02,200173.116752677160,202881.177706232610,6175
4,2014-01-03,326906.074176330640,333806.291873354810,10084


In [0]:
df.tail() # Shows last 5 rows

,Date,Sales,Cost Of Sales,Quantity Sold
1048,2016-11-12,164998.844641164880,156263.149473909730,3843
1049,2016-11-13,97946.783047396821,93112.608909273331,2281
1050,2016-11-14,87834.253684381343,83068.269026507172,2046
1051,2016-11-15,95509.134975070017,88104.204530400457,2181
1052,2016-11-16,77229.971885239516,71428.870002172305,1763


# Inspect data

In [0]:
df.shape # Shows number of rows and columns
# It gives us a quick understanding of the scale of the dataset.

(1053, 4)

In [0]:
df.info() #

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1053 entries, 0 to 1052
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           1053 non-null   datetime64[ns]
 1   Sales          1053 non-null   object        
 2   Cost Of Sales  1053 non-null   object        
 3   Quantity Sold  1053 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 33.0+ KB


In [0]:
df.isnull().sum()
# Check missing data
# Also prevents unreliable metrics

Date             0
Sales            0
Cost Of Sales    0
Quantity Sold    0
dtype: int64

In [0]:
# Returns a single True or False for the whole DataFrame
df.isnull().values.any()

# Returns True/False for each column
df.isnull().any()

Date             False
Sales            False
Cost Of Sales    False
Quantity Sold    False
dtype: bool

Dataset has no missing values in all four columns.

In [0]:
df.duplicated().sum() # Check for duplicates
# Also protects the accuracy of results

np.int64(0)

Dataset has no duplicate rows or columns

# Summary Statistics

In [0]:
df.describe() # Understand the statistical characteristics of the numerical variables.

# It will also help in terms of identifying average sales and average quantity.

,Date,Quantity Sold
count,1053,1053.000000
mean,2015-06-09 00:00:00,5014.123457
min,2013-12-30 00:00:00,508.000000
25%,2014-09-19 00:00:00,2046.000000
50%,2015-06-09 00:00:00,3843.000000
75%,2016-02-27 00:00:00,6781.000000
max,2016-11-16 00:00:00,26410.000000
std,NaN,3901.251594


# Feature Engineering
* This is where we create the metrics
* Financial metrics are rounded to two decimal places for readability and presentation.

# Created Daily Unit Price
Due to the dataset provides total sales and total units sold, but we need the selling price per unit.

Allow us to understand daily pricing and identify possible promotional periods.

In [0]:
# Question 1 what is the daily sales price per unit
df['Unit Price'] = df['Sales'] / df['Quantity Sold']
df.head()

,Date,Sales,Cost Of Sales,Quantity Sold,Unit Price
0,2013-12-30,223937.967869726530,230079.621012448040,6827,32.80181161120939358429764172
1,2013-12-31,300345.484563985660,306986.120484283190,9268,32.40672038886336426413465688
2,2014-01-01,86782.467727417796,87986.318214449886,2678,32.40570116781844510828976848
3,2014-01-02,200173.116752677160,202881.177706232610,6175,32.41669906925945910931174089
4,2014-01-03,326906.074176330640,333806.291873354810,10084,32.41829375013195557318524395


The sales, cost of sales and the unit price contained a lot of decimal numbers.

In [0]:
df['Unit Price'] = (df['Sales'] / df['Quantity Sold']).round(2)
df['Sales'] = df['Sales'].round(2)
if 'Cost Of Sales' in df.columns:
    df['Cost Of Sales'] = df['Cost Of Sales'].round(2)

display(
    df.head().style.format({
        'Sales': '{:,.2f}',
        'Cost Of Sales': '{:,.2f}',
        'Unit Price': '{:,.2f}'
    })
)

,Date,Sales,Cost Of Sales,Quantity Sold,Unit Price
0,2013-12-30 00:00:00,"223,937.97","230,079.62",6827,32.80
1,2013-12-31 00:00:00,"300,345.48","306,986.12",9268,32.41
2,2014-01-01 00:00:00,"86,782.47","87,986.32",2678,32.41
3,2014-01-02 00:00:00,"200,173.12","202,881.18",6175,32.42
4,2014-01-03 00:00:00,"326,906.07","333,806.29",10084,32.42


# Average Unit Sales Price
So that we know what amount the product normally sells for

Allows us to compare individual days against the overall average price.

In [0]:
# Question 2 
df['Unit Price'].mean().round(2)

np.float64(37.07)

The average unit sales price is R37.07 meaning thats the amount a product would normally sell for

# Gross Profit
Supporting question 3 and 4

In [0]:
# Calculate Gross Profit
df['Gross Profit'] = df['Sales'] - df['Cost Of Sales']

# Question 3: Calculate Daily Gross Profit %
df['Gross Profit %'] = (df['Gross Profit'] / df['Sales']) * 100

# Question 4: Calculate Daily Gross Profit Per Unit (in Rands and %)
df['Gross Profit Per Unit (R)'] = df['Gross Profit'] / df['Quantity Sold']
df['% Gross Profit Per Unit'] = (df['Gross Profit Per Unit (R)'] / df['Unit Price']) * 100

# Round the calculated metrics for presentation
df['Gross Profit'] = df['Gross Profit'].round(2)
df['Gross Profit %'] = df['Gross Profit %'].round(2)
df['Gross Profit Per Unit (R)'] = df['Gross Profit Per Unit (R)'].round(2)
df['% Gross Profit Per Unit'] = df['% Gross Profit Per Unit'].round(2)

display(
    df[['Date', 'Unit Price', 'Quantity Sold', 'Gross Profit', 
        'Gross Profit %', 'Gross Profit Per Unit (R)', '% Gross Profit Per Unit']].head().style.format({
        'Unit Price': 'R{:,.2f}',
        'Gross Profit': 'R{:,.2f}',
        'Gross Profit %': '{:,.2f}%',
        'Gross Profit Per Unit (R)': 'R{:,.2f}',
        '% Gross Profit Per Unit': '{:,.2f}%'
    }).set_table_styles(
        [{'selector': 'th, td', 'props': [('padding', '12px')]}]
    )
)

,Date,Unit Price,Quantity Sold,Gross Profit,Gross Profit %,Gross Profit Per Unit (R),% Gross Profit Per Unit
0,2013-12-30 00:00:00,R32.80,6827,"R-6,141.65",-2.74%,R-0.90,-2.74%
1,2013-12-31 00:00:00,R32.41,9268,"R-6,640.64",-2.21%,R-0.72,-2.21%
2,2014-01-01 00:00:00,R32.41,2678,"R-1,203.85",-1.39%,R-0.45,-1.39%
3,2014-01-02 00:00:00,R32.42,6175,"R-2,708.06",-1.35%,R-0.44,-1.35%
4,2014-01-03 00:00:00,R32.42,10084,"R-6,900.22",-2.11%,R-0.68,-2.11%


# Pricing Analysis
To see how the selling price changes from day to day.

normal pricing

price reductions

unusually high prices

potential promotional periods


To compare daily prices against the typical price.

Makes potential price reductions easier to identify.

In [0]:
# Question 2 and 5
average_price = df['Unit Price'].mean()

fig = px.line(
    df,
    x='Date',
    y='Unit Price',
    title='Daily Unit Price vs Average Unit Price',
    labels={
        'Date': 'Date',
        'Unit Price': 'Unit Price (R)'
    }
)

fig.add_hline(
    y=average_price,
    line_dash='dash',
    annotation_text=f'Average Price: R{average_price:.2f}'
)

fig.show()

The line graph shows the daily unit price frequently moving above and below the average price of R37.07. From 4 November 2014 to 23 August 2015, the daily unit price remained above the average, indicating that the product was selling at a higher price than its average. After 23 August 2015, the daily unit price began fluctuating more frequently, moving both above and below the average price.

The frequent movement above and below the average indicates that the product's selling price varied over time. The graph does not explain the reasons for these price changes, so further analysis of quantity sold and promotional periods is required.

# Profitability Analysis
Track how profitability changes from day to day.

This will help identify:

high margin periods

low margin periods

unusual profitability changes

# Gross Profit % Over 
Track how the percentage of sales kept as profit changes from day to day.

Help identify periods with higher or lower profit margins.

In [0]:
# make sure the profitability columns have been calculated:

# Calculate Gross Profit
df['Gross Profit'] = df['Sales'] - df['Cost Of Sales']

# Calculate Gross Profit %
df['Gross Profit %'] = (df['Gross Profit'] / df['Sales']) * 100

# Calculate Gross Profit Per Unit
df['Gross Profit Per Unit'] = df['Gross Profit'] / df['Quantity Sold']

In [0]:
# Q3 What is the daily % gross profit
import plotly.express as px

fig = px.line(
    df,
    x='Date',
    y='Gross Profit %',
    title='Daily Gross Profit % Over Time',
    labels={
        'Date': 'Date',
        'Gross Profit %': 'Gross Profit (%)'
    }
)

fig.show()

The graph shows that the daily gross profit percentage frequently moved between positive and negative values. Positive values indicate that the business made a gross profit, while negative values indicate that the business experienced a gross loss. The highest gross profit percentage was 14.41% on 11 July 2016, while the lowest was -14.85% on 29 November 2015. This indicates that the business experienced significant changes in profitability across different days.

# Gross Profit Per Unit Over Time
To see how much profit was generated from each unit sold on each day.

Help identify periods when the profit earned per unit was higher or lower.

In [0]:
# Q4 What is the daily% gross profit per unit?
fig = px.line(
    df,
    x='Date',
    y='Gross Profit Per Unit',
    title='Daily Gross Profit Per Unit Over Time',
    labels={
        'Date': 'Date',
        'Gross Profit Per Unit': 'Gross Profit Per Unit (R)'
    }
)

fig.show()

The graph shows that the daily gross profit generated per unit fluctuated significantly over time, frequently moving between positive and negative values. The highest daily gross profit per unit was R6.08 on 26 November 2014, while the lowest was -R4.96 on 29 November 2015. This indicates that the amount of profit generated from each unit varied significantly across different days, with some days generating a profit per unit and others resulting in a loss per unit.

# Promotional & Demand Analysis

# Daily Sales Trend
Understand how total sales change over time.

It will help identify:

sales growth

decrease

spikes

unusual periods

In [0]:
# Q6 additional insights
# provides supporting context for Q5
fig = px.line(
    df,
    x='Date',
    y='Sales',
    title='Daily Sales Trend',
    labels={
        'Date': 'Date',
        'Sales': 'Sales (R)'
    }
)

fig.show()

The graph shows the daily sales trend from January 2014 to November 2016, with sales frequently fluctuating between higher and lower values across different days. The highest daily sales were recorded on 1 March 2014 at approximately R846.67K, while the lowest daily sales were recorded on 20 August 2015 at approximately R24.75K. This indicates that the business experienced significant variations in daily sales over the period.

# Daily Quantity Sold
To understand how many units were sold each day.

Help us see changes in customer demand and later compare quantity sold during promotional periods with normal periods.

In [0]:
# Answers Q5 
# A. Price Elasticity of demand
# B. Promotional performance
fig = px.line(
    df,
    x='Date',
    y='Quantity Sold',
    title='Daily Quantity Sold',
    labels={
        'Date': 'Date',
        'Quantity Sold': 'Quantity Sold (Units)'
    }
)

fig.show()

The graph shows the daily quantity of units sold from January 2014 to November 2016, with the quantity sold fluctuating between higher and lower values across different days. The highest quantity sold was recorded on 1 March 2014 at approximately 26.41K units, while the lowest was recorded on 4 September 2016 at approximately 508 units. This indicates that the business experienced significant variations in the quantity of units sold over the period.

# Conclusion of both graphs
The Daily Sales Trend and Daily Quantity Sold graphs both show significant fluctuations from January 2014 to November 2016. Both sales and quantity sold frequently moved between higher and lower values across different days. The highest sales and highest quantity sold occurred on 1 March 2014, while lower values occurred on other days. This indicates that the business experienced significant changes in both sales performance and customer demand over time. However, these graphs alone do not explain what caused these changes, so further analysis of price and promotional periods is required.

# Identified 3 promotional periods
To find three periods where the product may have been sold at a promotional or special price so that we can analyse customer demand during those periods.

It allows us to compare price changes with changes in quantity sold and determine whether customers responded to the lower price.

## Question 5: Analysis of 3 Distinct Promotional Periods & Price Elasticity of Demand (PED)

Using the overall average unit price of **R37.07** as a reference, we establish a **Normal Baseline Period** (`Unit Price >= R37.07`, averaging **R40.14** per unit and **3,179.49** units sold per day).

We then isolate **three distinct promotional windows** across different years where prices dropped significantly:
1. **Promotional Period 1 (New Year Special):** 30 Dec 2013 – 07 Jan 2014 (9 days)
2. **Promotional Period 2 (Spring Promotion):** 24 Sep 2015 – 06 Oct 2015 (13 days)
3. **Promotional Period 3 (Mid-Year Winter Sale):** 23 Jun 2016 – 01 Jul 2016 (9 days)

In [0]:
# Ensure exact unrounded calculations
df['Exact_Unit_Price'] = df['Sales'] / df['Quantity Sold']
df['Exact_GP_Pct'] = ((df['Sales'] - df['Cost Of Sales']) / df['Sales']) * 100

# Normal Baseline Period (Unit Price >= R37.07)
normal_period = df[df['Exact_Unit_Price'] >= 37.07]
avg_normal_price = normal_period['Exact_Unit_Price'].mean()
avg_normal_qty = normal_period['Quantity Sold'].mean()

# Define the 3 distinct promotional windows required by the rubric
promo_windows = [
    ("Promo 1: New Year Special", "2013-12-30", "2014-01-07"),
    ("Promo 2: Spring Promotion", "2015-09-24", "2015-10-06"),
    ("Promo 3: Mid-Year Winter Sale", "2016-06-23", "2016-07-01")
]

results = []

for name, start_date, end_date in promo_windows:
    promo_df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]
    
    avg_promo_price = promo_df['Exact_Unit_Price'].mean()
    avg_promo_qty = promo_df['Quantity Sold'].mean()
    avg_promo_gp_pct = promo_df['Exact_GP_Pct'].mean()
    
    # Percentage changes relative to normal baseline
    pct_change_qty = (avg_promo_qty - avg_normal_qty) / avg_normal_qty
    pct_change_price = (avg_promo_price - avg_normal_price) / avg_normal_price
    
    # Price Elasticity of Demand (PED) = (% Change in Qty) / (% Change in Price)
    ped = pct_change_qty / pct_change_price
    
    results.append({
        "Promotional Period": name,
        "Date Range": f"{start_date} to {end_date}",
        "Avg Promo Price": round(avg_promo_price, 2),
        "Price Change (%)": round(pct_change_price * 100, 2),
        "Avg Daily Qty": round(avg_promo_qty, 2),
        "Qty Change (%)": round(pct_change_qty * 100, 2),
        "PED": round(ped, 2),
        "Avg Gross Profit (%)": round(avg_promo_gp_pct, 2)
    })

ped_summary_df = pd.DataFrame(results)
display(ped_summary_df)

Promotional Period,Date Range,Avg Promo Price,Price Change (%),Avg Daily Qty,Qty Change (%),PED,Avg Gross Profit (%)
Promo 1: New Year Special,2013-12-30 to 2014-01-07,32.46,-19.12,7376.11,131.99,-6.9,-2.1
Promo 2: Spring Promotion,2015-09-24 to 2015-10-06,33.38,-16.83,8425.08,164.98,-9.8,-13.66
Promo 3: Mid-Year Winter Sale,2016-06-23 to 2016-07-01,33.39,-16.8,8797.44,176.69,-10.52,-8.24


### Question 5(a): Price Elasticity of Demand (PED) Results
Compared against the normal baseline average price of **R40.14** and normal daily demand of **3,179.49 units**, all three promotional periods demonstrate **highly elastic demand** ($\text{PED} < -1$):
* **Promo 1 (New Year Special):** Price dropped by **19.12%** (to R32.46), driving a **131.99%** surge in quantity sold. **PED = -6.90**.
* **Promo 2 (Spring Promotion):** Price dropped by **16.83%** (to R33.38), driving a **164.98%** surge in quantity sold. **PED = -9.80**.
* **Promo 3 (Mid-Year Winter Sale):** Price dropped by **16.80%** (to R33.39), driving a **176.69%** surge in quantity sold. **PED = -10.52**.

---

### Question 5(b): Does the Product Perform Better or Worse at a Promotional Price?
In my opinion, the product performs **better in volume and customer demand**, but **worse in standalone gross profitability**:
* **Volume Performance (Better):** Because demand is highly elastic ($\text{PED}$ between **-6.90** and **-10.52**), offering a ~17%–19% discount more than doubles daily unit sales, making promotions extremely effective for driving store traffic and clearing inventory.
* **Profitability Performance (Worse):** During all three promotional windows, the average Gross Profit % turned negative (**-2.10%**, **-13.66%**, and **-8.24%** respectively). Because the promotional prices fell below the Cost of Sales per unit, selling higher volumes generated a gross loss per unit. Therefore, promotions are only beneficial if used as a strategic loss-leader to attract customers who purchase other profitable items.

# Question 6 Additional Insights
Key Performance Indicator/ metrics

KPI is an important number that helps a business measure how well it is performing

# Total Sales

To determine the total sales generated by the business across the entire dataset.

Provides an overall measure of the business's sales performance.

In [0]:
total_sales = df['Sales'].sum()

print(f"Total Sales: R{total_sales:,.2f}")

Total Sales: R186,909,828.26


The total sales generated by the business across the entire dataset were R186,909,828.26. This provides an overall measure of the business's sales performance during the period analysed.

# Total Quantity Sold

To determine the total number of units sold during the entire period.

Provides an overall measure of customer demand and product sales volume.

In [0]:
total_quantity = df['Quantity Sold'].sum()

print(f"Total Quantity Sold: {total_quantity:,.2f} units")

Total Quantity Sold: 5,279,872.00 units


The business sold a total of 5,279,872 units during the period analysed. This indicates a substantial volume of product demand and provides an overall measure of the number of units sold by the business.

# Total Gross Profit

To determine how much gross profit the business generated after subtracting the cost of sales.

Provides an overall view of the business's profitability across the entire dataset.

In [0]:
total_gross_profit = (df['Sales'] - df['Cost Of Sales']).sum()

print(f"Total Gross Profit: R{total_gross_profit:,.2f}")

Total Gross Profit: R-7,116,281.19


In [0]:
print(f"Total Sales: R{df['Sales'].sum():,.2f}")
print(f"Total Cost of Sales: R{df['Cost Of Sales'].sum():,.2f}")
print(f"Total Gross Profit: R{(df['Sales'].sum() - df['Cost Of Sales'].sum()):,.2f}")

Total Sales: R186,909,828.26
Total Cost of Sales: R194,026,109.45
Total Gross Profit: R-7,116,281.19


The business generated total sales of R186.91 million, while the total cost of sales was R194.03 million. This resulted in an overall gross loss of R7.12 million, indicating that the cost of sales exceeded the revenue generated from sales during the period analysed.


This also tells management that high sales revenue did not translate into overall gross profitability. The business should therefore investigate its pricing and cost of sales, especially because your earlier analysis showed that the unit price frequently moved above and below the typical price.

# Monthly Sales Performance
To understand how sales performed over longer periods by grouping daily sales into months.

Makes it easier to identify longer term sales trends, increases, decreases, and periods of unusually high or low sales.

In [0]:
# Create monthly sales
monthly_sales = (
    df.groupby(df['Date'].dt.to_period('M'))['Sales']
    .sum()
    .reset_index()
)

# Convert month back to datetime
monthly_sales['Date'] = monthly_sales['Date'].dt.to_timestamp()

# Create line chart
fig = px.line(
    monthly_sales,
    x='Date',
    y='Sales',
    title='Monthly Sales Performance',
    labels={
        'Date': 'Month',
        'Sales': 'Sales (R)'
    }
)

fig.show()

The monthly sales performance shows significant fluctuations throughout the period analysed. Sales reached several high points, particularly in March 2014 with R8.27 million, August 2014 with R10.01 million, and May 2015 with R8.38 million. However, sales also experienced noticeable declines, with a downward trend toward the end of the period. This indicates that the business's sales performance was not consistent across the months.

Overall, the analysis shows that the product experienced significant fluctuations in price, profitability, sales and customer demand throughout the period analysed. The typical selling price was R37.07, while the daily unit price frequently moved above and below this level, showing that the selling price varied over time.

Profitability also varied considerably. The daily gross profit percentage ranged from -14.85% to 14.41%, while gross profit per unit ranged from -R4.96 to R6.08. This shows that the business experienced both profitable and loss-making days and that profitability changed significantly over time.

The analysis of the three promotional periods showed that customers were generally more responsive to lower promotional prices, with changes in price being associated with changes in the quantity sold. The Price Elasticity of Demand results helped demonstrate how sensitive demand was to price changes during each promotional period. Overall, the promotional analysis suggests that the product can perform better in terms of quantity sold when offered at a promotional price.

The additional analysis showed that the business generated R186.91 million in total sales and sold approximately 5.28 million units. However, total Cost of Sales was R194.03 million, resulting in an overall gross loss of R7.12 million. This indicates that high sales revenue and sales volume did not translate into overall gross profitability during the period analysed.

Overall, the findings suggest that pricing and promotions are important factors associated with customer demand, but the business should also pay close attention to its Cost of Sales and pricing strategy. Increasing sales volume alone is not enough; the business also needs to ensure that its selling prices and costs allow it to generate sustainable gross profit.